# AI & ML Task 3
## Model Validation, Overfitting Control & Hyperparameter Tuning

## 1. Project Overview

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split,cross_val_score,GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression,Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error,r2_score
import joblib

## 2. Load Dataset

In [ ]:
data=fetch_california_housing(as_frame=True)
df=pd.concat([data.data,data.target.rename('HousePrice')],axis=1)
df.head()

## 3. EDA

In [ ]:
print(df.shape)
print(df.info())
print(df.describe())
print(df.isnull().sum())

## 4. Correlation Heatmap

In [ ]:
plt.figure(figsize=(12,8))
sns.heatmap(df.corr(),annot=True,cmap='coolwarm')
plt.show()

## 5. Feature Scaling

In [ ]:
X=df.drop('HousePrice',axis=1)
y=df['HousePrice']
scaler=StandardScaler()
X_scaled=scaler.fit_transform(X)

## 6. Train Test Split

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X_scaled,y,test_size=0.2,random_state=42)

## 7. Baseline Models

In [ ]:
linear_model=LinearRegression()
ridge_model=Ridge(alpha=1.0)
tree_model=DecisionTreeRegressor(random_state=42)

linear_model.fit(X_train,y_train)
ridge_model.fit(X_train,y_train)
tree_model.fit(X_train,y_train)

## 8. Overfitting Detection

In [ ]:
train_pred=tree_model.predict(X_train)
test_pred=tree_model.predict(X_test)

train_rmse=np.sqrt(mean_squared_error(y_train,train_pred))
test_rmse=np.sqrt(mean_squared_error(y_test,test_pred))

print(train_rmse,test_rmse)

In [ ]:
plt.bar(['Train RMSE','Test RMSE'],[train_rmse,test_rmse])
plt.title('Overfitting Analysis')
plt.show()

## 9. Cross Validation

In [ ]:
cv_scores=cross_val_score(tree_model,X_scaled,y,scoring='neg_root_mean_squared_error',cv=5)
cv_rmse=-cv_scores.mean()
print(cv_rmse)

## 10. Hyperparameter Tuning

In [ ]:
param_grid={
'max_depth':[3,5,7,10],
'min_samples_split':[2,5,10]
}

grid=GridSearchCV(
DecisionTreeRegressor(random_state=42),
param_grid,
scoring='neg_root_mean_squared_error',
cv=5)

grid.fit(X_train,y_train)
print(grid.best_params_)

## 11. Tuned Model Evaluation

In [ ]:
best_tree=grid.best_estimator_
y_pred=best_tree.predict(X_test)

rmse=np.sqrt(mean_squared_error(y_test,y_pred))
r2=r2_score(y_test,y_pred)

print(rmse,r2)

## 12. Model Comparison Table

In [ ]:
lr_pred=linear_model.predict(X_test)
ridge_pred=ridge_model.predict(X_test)

results=pd.DataFrame({
'Model':['Linear Regression','Ridge Regression','Tuned Decision Tree'],
'RMSE':[np.sqrt(mean_squared_error(y_test,lr_pred)),
np.sqrt(mean_squared_error(y_test,ridge_pred)),
rmse],
'R2 Score':[r2_score(y_test,lr_pred),
r2_score(y_test,ridge_pred),
r2]
})
results

## 13. Model Comparison Charts

In [ ]:
sns.barplot(data=results,x='Model',y='RMSE')
plt.title('RMSE Comparison')
plt.show()

sns.barplot(data=results,x='Model',y='R2 Score')
plt.title('R2 Comparison')
plt.show()

## 14. Actual vs Predicted

In [ ]:
plt.scatter(y_test,y_pred,alpha=0.4)
plt.plot([min(y_test),max(y_test)],[min(y_test),max(y_test)],color='red')
plt.show()

## 15. Residual Analysis

In [ ]:
residuals=y_test-y_pred
sns.histplot(residuals,kde=True)
plt.show()

plt.scatter(y_pred,residuals,alpha=0.4)
plt.axhline(y=0,color='red',linestyle='--')
plt.show()

## 16. Save Model

In [ ]:
joblib.dump(best_tree,'task3_best_model.joblib')

## 17. Conclusion